In [20]:
from concurrent.futures import ThreadPoolExecutor
import time
import requests
import pandas as pd
import psycopg2

In [21]:
from dotenv import load_dotenv
import os

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

print("Connected!")

Connected!


In [12]:
import re

def cleaning_name(name):
    name = re.sub(r'\b(llc|ltd|inc|corp|co|lp|llp|pvt|limited|corporation|)\b','',name,flags=re.IGNORECASE).lower()

    name = re.sub(r'[^a-z0-9]', '', name) 
    return name

def get_data_jobs(slug):
    # slug = cleaning_name(slug)
    url = f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
    res= requests.get(url)
    # us_keywords = ['usa', 'united states', 'united states of america', 'remote']


    if res.status_code!=200:
        return []
    jobs_list = res.json()['jobs']
    matches=[]
    for i in jobs_list:
        if 'data' in i['title'].lower():
            matches.append({'company':slug,
                            'title':i['title'],
                            'location':i['location']['name'],
                            'url': i['absolute_url']

            
                            })
    return matches
    


In [17]:
df= pd.read_csv('confirmed_greenhouse_slugs.csv')
df.head(10)
df.shape

(126, 1)

In [18]:
confirmed = df.iloc[:, 0].tolist()

In [19]:
all_jobs = []
for slug in confirmed:
    results = get_data_jobs(slug)
    all_jobs.extend(results)

print(f"Total jobs found: {len(all_jobs)}")
  # print first 5 jobs for inspection

Total jobs found: 165
